# CIC-IDS2017 Conformal Signature–ML NIDS (Colab)

This notebook runs the real-data path on one Wednesday CIC-IDS2017 capture. It downloads files from a pinned **unofficial** Hugging Face mirror, verifies the mirror object SHA-256 values, extracts the original `GeneratedLabelledFlows` CSV, replays the PCAP with Suricata, aligns alerts, trains/calibrates the pipeline, and saves auditable outputs.

The mirror maintainer says it is not the original distribution. For publication, prefer the official CIC/UNB source when available and cite the CIC-IDS2017 paper. The mirror hashes prove file integrity for the pinned mirror revision, not publisher authenticity. The PCAP is about 13.4 GB; ensure the Colab runtime has enough disk and time.


In [ ]:
!apt-get update -qq && apt-get install -y -qq suricata
!python -m pip install -q huggingface_hub numpy pandas scikit-learn
!suricata -V


In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, zipfile
from huggingface_hub import hf_hub_download

REPO = "bencorn/CICIDS2017"
REVISION = "c5637a4d7582337ef60cfb20b77227fbfe687ed6"
EXPECTED = {
    "csvs/GeneratedLabelledFlows.zip": "7bdbef286f8893f31c6db12105fa097fa5c2dcc6733179037a08129d150ea27a",
    "pcaps/Wednesday-workingHours.pcap": "cd2674db7559a53f24bc03be3239b315700174ccaef72d10f5edc4c1a08f6186",
}

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

paths = {}
for filename, expected_sha in EXPECTED.items():
    local = Path(hf_hub_download(repo_id=REPO, repo_type="dataset", filename=filename, revision=REVISION))
    actual = sha256(local)
    print(filename, local.stat().st_size, actual)
    if actual != expected_sha:
        raise RuntimeError(f"SHA-256 mismatch for {filename}: {actual}")
    paths[filename] = local

PCAP = paths["pcaps/Wednesday-workingHours.pcap"]


In [ ]:
import zipfile

FLOW_CSV = Path("/content/Wednesday-workingHours.pcap_ISCX.csv")
member = None
with zipfile.ZipFile(paths["csvs/GeneratedLabelledFlows.zip"]) as archive:
    for name in archive.namelist():
        if Path(name).name == FLOW_CSV.name:
            member = name
            break
    if member is None:
        raise FileNotFoundError(f"{FLOW_CSV.name} not found in GeneratedLabelledFlows.zip")
    with archive.open(member) as source, FLOW_CSV.open("wb") as target:
        shutil.copyfileobj(source, target)
print("Extracted", member, "bytes=", FLOW_CSV.stat().st_size)

import pandas as pd
preview = pd.read_csv(FLOW_CSV, usecols=lambda name: name.strip() in {
    "Timestamp", "Source IP", "Source Port", "Destination IP", "Destination Port", "Protocol", "Label"
})
print("Columns:", list(preview.columns))
print("Raw timestamp range:", preview["Timestamp"].min(), "to", preview["Timestamp"].max())
print("Labels:", preview["Label"].value_counts(dropna=False).head(20).to_dict())


In [ ]:
from pathlib import Path

REPO_DIR = Path("/content/conformal-nids-cascade-demo")
PROJECT_REVISION = "970c94da414e27680b5b2fdf880efc4230a6583f"
PROJECT_URL = "https://github.com/Nithinreddy3093/conformal-nids-cascade-demo.git"
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "init", str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "remote", "add", "origin", PROJECT_URL], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", PROJECT_REVISION], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
actual_revision = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert actual_revision == PROJECT_REVISION
print("Pinned project commit:", actual_revision)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-research.txt")], check=True)
sys.path.insert(0, str(REPO_DIR / "research"))

# Update this only after confirming how the timezone-naive CICFlowMeter timestamps
# correspond to the PCAP/EVE timestamps. Do not assume UTC without checking.
FLOW_TIMEZONE = "UTC"
SURICATA_CONFIG = Path("/etc/suricata/suricata.yaml")
SURICATA_OUT = Path("/content/suricata-output")
RULESET = Path("/var/lib/suricata/rules")
subprocess.run(["suricata-update"], check=True)
subprocess.run(["suricata", "-T", "-c", str(SURICATA_CONFIG)], check=True)


In [ ]:
from replay_suricata import run_replay

replay_manifest = run_replay(
    pcap=PCAP,
    config=SURICATA_CONFIG,
    output_dir=SURICATA_OUT,
    binary="suricata",
    ruleset=RULESET,
)
EVE = Path(replay_manifest["eve_json"]["path"])
print("EVE alerts:", replay_manifest["eve_json"]["events"].get("alert", 0))
print("PCAP SHA-256:", replay_manifest["inputs"]["pcap"]["sha256"])


In [ ]:
from pipeline import run

RESULTS = Path("/content/results/wednesday-run")
report = run(
    csv_path=FLOW_CSV,
    eve_path=EVE,
    pcap_path=PCAP,
    replay_manifest_path=SURICATA_OUT / "suricata_run.json",
    out_dir=RESULTS,
    budget=5,
    window_seconds=900,
    alpha=0.10,
    tolerance=2.0,
    ruleset_path=RULESET,
    flow_timezone=FLOW_TIMEZONE,
    dataset_source=f"https://huggingface.co/datasets/{REPO}",
    dataset_revision=REVISION,
)
print(json.dumps({
    "split_rows": report["split_rows"],
    "suricata_alignment": report["suricata_alignment"],
    "metrics": report["metrics"],
    "conformal_diagnostics": report["conformal_diagnostics"],
}, indent=2))
if report["suricata_alignment"]["matched_alert_events"] == 0:
    raise RuntimeError("No Suricata alerts aligned to labeled flows. Check timezone, rules, and capture/CSV pairing before interpreting the run.")


In [ ]:
import zipfile

artifact = Path("/content/wednesday-run-artifacts.zip")
with zipfile.ZipFile(artifact, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in RESULTS.iterdir():
        if path.is_file():
            bundle.write(path, arcname=path.name)
    bundle.write(SURICATA_OUT / "suricata_run.json", arcname="suricata_run.json")
print("Created:", artifact, "bytes=", artifact.stat().st_size)
from google.colab import files
files.download(str(artifact))
